# 3장 1강: 교차표와 카이제곱 독립성 검정 이론 — 실습문제

## 실습 목표

- 두 범주형 변수의 교차표를 구성하고 관측빈도를 해석할 수 있다.
- 행 합계와 열 합계를 이용해 기대빈도를 직접 계산할 수 있다.
- 관측빈도와 기대빈도로 카이제곱 통계량과 자유도를 계산할 수 있다.
- 카이제곱 독립성 검정을 수행하고 기대빈도 조건을 확인할 수 있다.
- p-value와 범주별 비율을 함께 사용하여 변수 간 관련성을 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas, NumPy
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `YearBuilt` | 건축연도 |
| `CentralAir` | 중앙 냉방시설 유무 |
| `KitchenQual` | 주방 품질 |
| `PavedDrive` | 진입로 포장 상태 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> `chi2_contingency()`에서는 강의자료와 동일하게 `correction=False`를 사용합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# 1. 필요한 라이브러리 불러오기 (위 import문)

# 2. 데이터 불러오기 (동일 폴더 또는 data/ 폴더 순차 탐색)
ROOT = Path.cwd()
data_path = ROOT / "ames_housing.csv"
if not data_path.exists():
    data_path = ROOT / "data" / "ames_housing.csv"

df = pd.read_csv(data_path, encoding="utf-8-sig")

# 3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행 확인
print("=" * 50)
print(f"데이터 크기 (행, 열): {df.shape}")
print("-" * 50)
print("컬럼별 결측치 수:")
print(df.isnull().sum())
print("-" * 50)
print("컬럼 목록:")
print(df.columns.tolist())
print("-" * 50)
print("상위 5개 행:")
print(df.head())
print("=" * 50)


데이터 크기 (행, 열): (1460, 10)
--------------------------------------------------
컬럼별 결측치 수:
SalePrice       0
GrLivArea       0
LotArea         0
OverallQual     0
KitchenQual     0
CentralAir      0
HeatingQC       0
PavedDrive      0
Neighborhood    0
YearBuilt       0
dtype: int64
--------------------------------------------------
컬럼 목록:
['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']
--------------------------------------------------
상위 5개 행:
   SalePrice  GrLivArea  LotArea  OverallQual KitchenQual CentralAir  \
0     208500       1710     8450            7          Gd          Y   
1     181500       1262     9600            6          TA          Y   
2     223500       1786    11250            7          Gd          Y   
3     140000       1717     9550            7          Gd          Y   
4     250000       2198    14260            8          Gd          Y   

  HeatingQC PavedDrive Neighbo

---

## 필수 1. 교차표와 카이제곱 통계량 직접 계산

### 문제 1-1. 주택 품질 구간과 중앙 냉방시설의 관계

#### 문제 설명

`OverallQual`을 세 구간으로 나눈 뒤 중앙 냉방시설 유무와의 관계를 확인합니다.

- 낮음: 1~5점
- 보통: 6~7점
- 높음: 8~10점

#### 요구사항

1. `pd.cut()`을 이용해 위 기준으로 `QualityGroup`을 만드세요.
2. 행에는 `QualityGroup`, 열에는 `CentralAir`가 오도록 합계 없는 교차표를 만드세요.
3. `margins=True`인 교차표도 별도로 만들어 행·열 합계를 확인하세요.
4. 합계가 없는 교차표를 NumPy 배열 `observed`로 변환하세요.
5. 행 합계, 열 합계, 전체 합계를 계산하세요.
6. `(행 합계 × 열 합계) / 전체 합계`로 기대빈도를 직접 계산하세요.
7. `Σ(관측-기대)²/기대`로 카이제곱 통계량을 직접 계산하세요.
8. `(행 수-1) × (열 수-1)`로 자유도를 계산하세요.
9. 모든 기대빈도가 5 이상인지 확인하세요.
10. `stats.chi2_contingency(..., correction=False)` 결과와 직접 계산한 값을 비교하세요.
11. p-value를 이용해 두 변수가 관련 있는지 판단하세요.

#### 해석 질문

**Q1.** 교차표 각 칸의 숫자는 무엇을 의미하나요?  
**Q2.** 기대빈도는 어떤 가정 아래 계산되는 값인가요?  
**Q3.** 직접 계산한 카이제곱 통계량과 함수의 결과는 일치하나요?  
**Q4.** 주택 품질 구간과 중앙 냉방시설 유무는 서로 독립이라고 볼 수 있나요?

#### 제출 결과

- 관측빈도 교차표와 주변합
- 기대빈도
- 수동 계산한 카이제곱 통계량과 자유도
- 기대빈도 조건 확인
- 함수 결과와의 비교
- 독립성 판단
- Q1~Q4 답변

In [2]:
# 필수 1 코드를 작성하세요.

# 1. QualityGroup 생성 (1~5=낮음, 6~7=보통, 8~10=높음)
df["QualityGroup"] = pd.cut(
    df["OverallQual"],
    bins=[0, 5, 7, 10],
    labels=["낮음", "보통", "높음"]
)

# 2. 합계 없는 교차표 (행: QualityGroup, 열: CentralAir)
crosstab_no_margin = pd.crosstab(df["QualityGroup"], df["CentralAir"])
print("=" * 60)
print("[요구사항 2] 교차표 (합계 없음)")
print(crosstab_no_margin)

# 3. margins=True 교차표
crosstab_margin = pd.crosstab(df["QualityGroup"], df["CentralAir"], margins=True)
print("-" * 60)
print("[요구사항 3] 교차표 (행·열 합계 포함)")
print(crosstab_margin)

# 4. observed: NumPy 배열로 변환
observed = crosstab_no_margin.values
print("-" * 60)
print("[요구사항 4] observed (NumPy 배열)")
print(observed)

# 5. 행 합계, 열 합계, 전체 합계
row_sums = observed.sum(axis=1)
col_sums = observed.sum(axis=0)
total = observed.sum()
print("-" * 60)
print(f"[요구사항 5] 행 합계: {row_sums}")
print(f"[요구사항 5] 열 합계: {col_sums}")
print(f"[요구사항 5] 전체 합계: {total}")

# 6. 기대빈도 직접 계산: (행 합계 x 열 합계) / 전체 합계
expected = np.outer(row_sums, col_sums) / total
print("-" * 60)
print("[요구사항 6] 기대빈도 (직접 계산)")
print(np.round(expected, 4))

# 7. 카이제곱 통계량 직접 계산: Σ(관측-기대)²/기대
chi2_manual = ((observed - expected) ** 2 / expected).sum()
print("-" * 60)
print(f"[요구사항 7] 직접 계산한 카이제곱 통계량: {chi2_manual:.6g}")

# 8. 자유도: (행 수-1) x (열 수-1)
dof = (observed.shape[0] - 1) * (observed.shape[1] - 1)
print(f"[요구사항 8] 자유도: {dof}")

# 9. 모든 기대빈도가 5 이상인지 확인
condition_ok = (expected >= 5).all()
print(f"[요구사항 9] 모든 기대빈도 >= 5 여부: {condition_ok}")

# 10. stats.chi2_contingency() 결과와 직접 계산 값 비교
chi2_func, p_value, dof_func, expected_func = stats.chi2_contingency(observed, correction=False)
print("-" * 60)
print("[요구사항 10] 함수 결과와 직접 계산 비교")
print(f"카이제곱 통계량 - 직접 계산: {chi2_manual:.6g} / 함수: {chi2_func:.6g}")
print(f"자유도         - 직접 계산: {dof} / 함수: {dof_func}")
print(f"p-value (함수) : {p_value:.6g}")

# 11. p-value로 관련성 판단
alpha = 0.05
print("-" * 60)
if p_value < alpha:
    print(f"[요구사항 11] p-value({p_value:.6g}) < {alpha} 이므로 귀무가설을 기각합니다.")
    print("[요구사항 11] 주택 품질 구간과 중앙 냉방시설 유무는 서로 관련이 있습니다.")
else:
    print(f"[요구사항 11] p-value({p_value:.6g}) >= {alpha} 이므로 귀무가설을 기각할 수 없습니다.")
    print("[요구사항 11] 주택 품질 구간과 중앙 냉방시설 유무는 독립성을 기각할 증거가 부족합니다.")


[요구사항 2] 교차표 (합계 없음)
CentralAir     N    Y
QualityGroup         
낮음            71  467
보통            23  670
높음             1  228
------------------------------------------------------------
[요구사항 3] 교차표 (행·열 합계 포함)
CentralAir     N     Y   All
QualityGroup                
낮음            71   467   538
보통            23   670   693
높음             1   228   229
All           95  1365  1460
------------------------------------------------------------
[요구사항 4] observed (NumPy 배열)
[[ 71 467]
 [ 23 670]
 [  1 228]]
------------------------------------------------------------
[요구사항 5] 행 합계: [538 693 229]
[요구사항 5] 열 합계: [  95 1365]
[요구사항 5] 전체 합계: 1460
------------------------------------------------------------
[요구사항 6] 기대빈도 (직접 계산)
[[ 35.0068 502.9932]
 [ 45.0925 647.9075]
 [ 14.9007 214.0993]]
------------------------------------------------------------
[요구사항 7] 직접 계산한 카이제곱 통계량: 65.0304
[요구사항 8] 자유도: 2
[요구사항 9] 모든 기대빈도 >= 5 여부: True
----------------------------------------------------------

### 필수 1 답변

- **Q1.** 각 칸은 해당 품질 구간과 냉방시설 범주를 동시에 만족하는 **주택 수(관측빈도)**입니다. 행·열 합계는 주변합이며 검정 입력에는 포함하지 않습니다.

- **Q2.** 두 변수가 독립이라는 H0 아래, 관측된 주변합을 고정해 기대하는 빈도입니다. E=(행 합계×열 합계)/전체 합계입니다.

- **Q3.** 둘 다 χ²=65.0304, 자유도 2로 일치합니다. correction=False를 사용해 수동 Pearson 통계량과 같은 조건으로 비교했습니다. 기대빈도 최솟값은 14.9007로 모두 5 이상입니다.

- **Q4.** p=7.565e-15<0.05이므로 독립이라는 H0를 기각합니다. 품질 구간과 냉방시설 유무 사이에 유의한 관련성이 있습니다.


---

## 필수 2. 카이제곱 독립성 검정과 비율 해석

### 문제 2-1. 건축연도 구간과 중앙 냉방시설의 관계

#### 문제 설명

건축연도를 다음 세 구간으로 나누고 중앙 냉방시설 설치 여부와 관련이 있는지 확인하세요.

- 1980년 이전
- 1980~1999년
- 2000년 이후

#### 요구사항

1. `pd.cut()`로 `YearBuiltGroup`을 만드세요.
2. `YearBuiltGroup`과 `CentralAir`의 교차표를 만드세요.
3. 다음 가설을 작성하세요.
   - H₀: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
   - H₁: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 기대빈도 중 5 미만인 칸의 개수를 확인하세요.
7. `pd.crosstab(..., normalize="index")`로 건축연도 구간별 냉방시설 비율을 계산하세요.
8. 검정 결과와 행 비율을 함께 이용해 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 문제는 적합도 검정과 독립성 검정 중 무엇을 사용해야 하나요?  
**Q2.** 자유도는 얼마이며 어떻게 계산되나요?  
**Q3.** 기대빈도 조건은 충족되나요?  
**Q4.** 건축연도 구간과 중앙 냉방시설 유무 사이에는 유의한 관련성이 있나요?  
**Q5.** 카이제곱 검정 결과만으로 건축연도가 냉방시설 설치의 원인이라고 결론 내릴 수 있나요?

#### 제출 결과

- 교차표와 가설
- 카이제곱 검정 결과
- 기대빈도 조건
- 행 비율
- 관련성 및 인과관계 해석
- Q1~Q5 답변

In [3]:
# 필수 2 코드를 작성하세요.

# 1. YearBuiltGroup 생성
year_bins = [df["YearBuilt"].min() - 1, 1979, 1999, df["YearBuilt"].max()]
df["YearBuiltGroup"] = pd.cut(
    df["YearBuilt"],
    bins=year_bins,
    labels=["1980년 이전", "1980~1999년", "2000년 이후"]
)

# 2. YearBuiltGroup과 CentralAir의 교차표
crosstab2 = pd.crosstab(df["YearBuiltGroup"], df["CentralAir"])
print("=" * 60)
print("[요구사항 2] 교차표")
print(crosstab2)

# 3. 가설 작성
print("-" * 60)
print("[요구사항 3] 가설")
print("H0: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.")
print("H1: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.")

# 4~5. 카이제곱 독립성 검정 수행 및 결과 출력
chi2_stat2, p_value2, dof2, expected2 = stats.chi2_contingency(crosstab2, correction=False)
print("-" * 60)
print(f"[요구사항 5] 카이제곱 통계량: {chi2_stat2:.6g}")
print(f"[요구사항 5] p-value: {p_value2:.6g}")
print(f"[요구사항 5] 자유도: {dof2}")
print("[요구사항 5] 기대빈도:")
print(np.round(expected2, 4))

# 6. 기대빈도 중 5 미만인 칸의 개수
n_below5 = (expected2 < 5).sum()
print("-" * 60)
print(f"[요구사항 6] 기대빈도 5 미만 칸 개수: {n_below5}")

# 7. 건축연도 구간별 냉방시설 비율 (행 비율)
row_prop2 = pd.crosstab(df["YearBuiltGroup"], df["CentralAir"], normalize="index")
print("-" * 60)
print("[요구사항 7] 건축연도 구간별 냉방시설 비율")
print(row_prop2)

# 8. 검정 결과와 행 비율을 함께 이용한 해석
alpha = 0.05
print("-" * 60)
if p_value2 < alpha:
    print(f"[요구사항 8] p-value({p_value2:.6g}) < {alpha} 이므로 귀무가설을 기각합니다.")
    print("[요구사항 8] 건축연도 구간과 중앙 냉방시설 유무는 유의한 관련이 있습니다.")
else:
    print(f"[요구사항 8] p-value({p_value2:.6g}) >= {alpha} 이므로 귀무가설을 기각할 수 없습니다.")
    print("[요구사항 8] 건축연도 구간과 중앙 냉방시설 유무는 독립성을 기각할 증거가 부족합니다.")


[요구사항 2] 교차표
CentralAir       N    Y
YearBuiltGroup         
1980년 이전        95  753
1980~1999년       0  224
2000년 이후         0  388
------------------------------------------------------------
[요구사항 3] 가설
H0: 건축연도 구간과 중앙 냉방시설 유무는 서로 독립이다.
H1: 건축연도 구간과 중앙 냉방시설 유무는 서로 관련이 있다.
------------------------------------------------------------
[요구사항 5] 카이제곱 통계량: 73.333
[요구사항 5] p-value: 1.19109e-16
[요구사항 5] 자유도: 2
[요구사항 5] 기대빈도:
[[ 55.1781 792.8219]
 [ 14.5753 209.4247]
 [ 25.2466 362.7534]]
------------------------------------------------------------
[요구사항 6] 기대빈도 5 미만 칸 개수: 0
------------------------------------------------------------
[요구사항 7] 건축연도 구간별 냉방시설 비율
CentralAir             N         Y
YearBuiltGroup                    
1980년 이전        0.112028  0.887972
1980~1999년      0.000000  1.000000
2000년 이후        0.000000  1.000000
------------------------------------------------------------
[요구사항 8] p-value(1.19109e-16) < 0.05 이므로 귀무가설을 기각합니다.
[요구사항 8] 건축연도 구간과 중앙 냉방시설 유무는 유의한 관련이 있습니다.


### 필수 2 답변

- **Q1.** 건축연도 구간과 냉방시설 유무라는 두 범주형 변수의 관련성을 묻기 때문에 카이제곱 **독립성 검정**입니다.

- **Q2.** 3행×2열이므로 (3−1)(2−1)=2입니다.

- **Q3.** 기대빈도 최솟값 14.5753, 5 미만 칸 0개이므로 실습 기준을 충족합니다. 관측빈도 0과 기대빈도 5 미만은 다른 조건입니다.

- **Q4.** χ²=73.3330, p=1.191e-16로 유의한 관련성이 있습니다. 냉방시설 Y 비율은 1980년 이전 88.80%, 1980~1999년 100.00%, 2000년 이후 100.00%입니다.

- **Q5.** 인과결론을 내릴 수 없습니다. 건축연도는 무작위 배정되지 않았고 주택 품질·위치·보수 이력 등 다른 요인이 관련될 수 있습니다.


---

## 과제. 주방 품질과 진입로 포장 상태의 관계

### 문제 3-1. 두 범주형 변수 재분류 후 독립성 검정

#### 문제 설명

기대빈도가 너무 작은 범주를 줄이기 위해 주방 품질과 진입로 포장 상태를 다음과 같이 재분류합니다.

- `KitchenGroup`
  - 우수: `Ex`, `Gd`
  - 보통 이하: `TA`, `Fa`
- `DriveGroup`
  - 완전 포장: `Y`
  - 미포장·부분포장: `N`, `P`

#### 요구사항

1. 위 기준으로 `KitchenGroup`과 `DriveGroup`을 만드세요.
2. 두 변수의 교차표를 작성하세요.
3. 두 변수가 독립이라는 귀무가설과 관련이 있다는 대립가설을 작성하세요.
4. 카이제곱 독립성 검정을 수행하세요.
5. 카이제곱 통계량, p-value, 자유도와 기대빈도를 출력하세요.
6. 모든 기대빈도가 5 이상인지 확인하세요.
7. 주방 품질 집단별 진입로 포장 비율을 계산하세요.
8. 검정 결과와 비율 차이를 함께 사용하여 두 변수의 관련성을 해석하세요.

#### 해석 질문

**Q1.** 이 과제에서 범주를 재분류한 이유는 무엇인가요?  
**Q2.** 기대빈도 조건은 충족되나요?  
**Q3.** 주방 품질 집단과 진입로 포장 상태는 서로 독립이라고 볼 수 있나요?  
**Q4.** 두 주방 품질 집단의 완전 포장 비율은 각각 얼마인가요?

#### 제출 결과

- 재분류 코드와 교차표
- 가설 설정
- 카이제곱 검정 결과
- 기대빈도 조건 확인
- 행 비율과 결과 해석
- Q1~Q4 답변

In [4]:
# 과제 코드를 작성하세요.

# 0. 재분류 전 원본 범주 확인
print("KitchenQual 값 확인:", sorted(df["KitchenQual"].unique()))
print("PavedDrive 값 확인:", sorted(df["PavedDrive"].unique()))

# 1. KitchenGroup, DriveGroup 생성
df["KitchenGroup"] = np.where(df["KitchenQual"].isin(["Ex", "Gd"]), "우수", "보통 이하")
df["DriveGroup"] = np.where(df["PavedDrive"] == "Y", "완전 포장", "미포장·부분포장")

# 2. 두 변수의 교차표
crosstab3 = pd.crosstab(df["KitchenGroup"], df["DriveGroup"])
print("=" * 60)
print("[요구사항 2] 교차표")
print(crosstab3)

# 3. 가설 작성
print("-" * 60)
print("[요구사항 3] 가설")
print("H0: 주방 품질 집단과 진입로 포장 상태는 서로 독립이다.")
print("H1: 주방 품질 집단과 진입로 포장 상태는 서로 관련이 있다.")

# 4~5. 카이제곱 독립성 검정 수행 및 결과 출력
chi2_stat3, p_value3, dof3, expected3 = stats.chi2_contingency(crosstab3, correction=False)
print("-" * 60)
print(f"[요구사항 5] 카이제곱 통계량: {chi2_stat3:.6g}")
print(f"[요구사항 5] p-value: {p_value3:.6g}")
print(f"[요구사항 5] 자유도: {dof3}")
print("[요구사항 5] 기대빈도:")
print(np.round(expected3, 4))

# 6. 모든 기대빈도가 5 이상인지 확인
condition_ok3 = (expected3 >= 5).all()
print("-" * 60)
print(f"[요구사항 6] 모든 기대빈도 >= 5 여부: {condition_ok3}")

# 7. 주방 품질 집단별 진입로 포장 비율 (행 비율)
row_prop3 = pd.crosstab(df["KitchenGroup"], df["DriveGroup"], normalize="index")
print("-" * 60)
print("[요구사항 7] 주방 품질 집단별 진입로 포장 비율")
print(row_prop3)

# 8. 검정 결과와 비율 차이를 함께 사용한 해석
alpha = 0.05
print("-" * 60)
if p_value3 < alpha:
    print(f"[요구사항 8] p-value({p_value3:.6g}) < {alpha} 이므로 귀무가설을 기각합니다.")
    print("[요구사항 8] 주방 품질 집단과 진입로 포장 상태는 유의한 관련이 있습니다.")
else:
    print(f"[요구사항 8] p-value({p_value3:.6g}) >= {alpha} 이므로 귀무가설을 기각할 수 없습니다.")
    print("[요구사항 8] 주방 품질 집단과 진입로 포장 상태는 독립성을 기각할 증거가 부족합니다.")


KitchenQual 값 확인: ['Ex', 'Fa', 'Gd', 'TA']
PavedDrive 값 확인: ['N', 'P', 'Y']
[요구사항 2] 교차표
DriveGroup    미포장·부분포장  완전 포장
KitchenGroup                 
보통 이하              100    674
우수                  20    666
------------------------------------------------------------
[요구사항 3] 가설
H0: 주방 품질 집단과 진입로 포장 상태는 서로 독립이다.
H1: 주방 품질 집단과 진입로 포장 상태는 서로 관련이 있다.
------------------------------------------------------------
[요구사항 5] 카이제곱 통계량: 48.2523
[요구사항 5] p-value: 3.74762e-12
[요구사항 5] 자유도: 1
[요구사항 5] 기대빈도:
[[ 63.6164 710.3836]
 [ 56.3836 629.6164]]
------------------------------------------------------------
[요구사항 6] 모든 기대빈도 >= 5 여부: True
------------------------------------------------------------
[요구사항 7] 주방 품질 집단별 진입로 포장 비율
DriveGroup    미포장·부분포장     완전 포장
KitchenGroup                    
보통 이하         0.129199  0.870801
우수            0.029155  0.970845
------------------------------------------------------------
[요구사항 8] p-value(3.74762e-12) < 0.05 이므로 귀무가설을 기각합니다.
[요구사항 8] 주방 품질 집단과 진입로 포장 상

### 과제 답변

- **Q1.** 희소한 범주를 의미가 유사한 범주로 합쳐 기대빈도를 확보하고 카이제곱 근사의 적절성을 높이기 위해서입니다. 유의한 결과를 만들 목적으로 사후에 임의 병합해서는 안 됩니다.

- **Q2.** 기대빈도 최솟값은 56.3836이며 모든 칸이 5 이상입니다.

- **Q3.** 연속성 보정 없는 Pearson 검정에서 χ²=48.2523, 자유도 1, p=3.748e-12로 독립성을 기각합니다. 두 변수는 유의하게 관련되어 있습니다.

- **Q4.** 우수: 666/686 = 97.08%, 보통 이하: 674/774 = 87.08%입니다. 우수 집단에서 완전 포장 비율이 높지만 인과효과를 뜻하지는 않습니다.


---

## 실습 마무리

1. 교차표에서 관측빈도와 기대빈도는 어떻게 다른가요?
2. 기대빈도는 어떤 공식으로 계산하나요?
3. 카이제곱 통계량이 커진다는 것은 무엇을 의미하나요?
4. 독립성 검정과 적합도 검정은 변수 개수와 질문에서 어떻게 다른가요?
5. 기대빈도가 5보다 작은 칸이 있다면 무엇을 고려해야 하나요?
### 마무리 답변

1. 관측빈도는 실제 센 주택 수, 기대빈도는 H0 아래 예측되는 주택 수이며 기대빈도는 소수일 수 있습니다.

2. Eij=(i행 합계×j열 합계)/전체 관측 수입니다.

3. 독립일 때 기대하는 빈도와 실제 빈도의 차이가 커진다는 뜻입니다. p값은 자유도에도 의존합니다.

4. 독립성 검정은 두 범주형 변수의 관련성, 적합도 검정은 하나의 범주형 변수 분포가 사전 지정 확률과 맞는지를 묻습니다.

5. 의미를 보존한 범주 병합, 표본 추가, 2×2 표의 Fisher 정확검정 등을 검토합니다. 기대빈도와 독립적 관측이라는 전제를 함께 확인합니다.
